# Setup environment for Zotero API

In [1]:
# conda install pyzotero
import sys
from linecache import cache
!conda install --yes -c conda-forge --prefix {sys.prefix} pyzotero python-dateutil # defaultdict

Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - conda-forge
 - defaults
Platform: osx-arm64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda



# All requested packages already installed.



# Connect to Zotero API

In [2]:
from pyzotero import zotero
import os

library_id = '6004137'
library_type = 'group'
api_key = os.environ.get("ZOTERO_API_KEY")

zot = zotero.Zotero(library_id, library_type, api_key)
# check if the connection is successful
sample_items = zot.top(limit=5)
# we've retrieved the latest five top-level items in our library
# we can print each item's item type and ID
for item in sample_items:
    print('Item Type: %s | Key: %s' % (item['data']['itemType'], item['data']['key']))

Item Type: journalArticle | Key: Z8UFMW2K
Item Type: journalArticle | Key: ZKVU7IFM
Item Type: conferencePaper | Key: E9L8HPS6
Item Type: conferencePaper | Key: 8BDG2CVM
Item Type: journalArticle | Key: IZALNCMW


In [18]:
all_items = zot.everything(zot.items())

In [4]:
print(len(all_items))

2788


# Utilities (always evaluate this cell)

In [5]:
from dateutil import parser

## Define a function that takes a date string and returns the year
def get_year_from_date(date_str):
    """Extracts the year from a zotero date string.
    examples:
    '2025' --> 2025
    '2025-01-01' --> 2025
    '2025-01' --> 2025
    '2025 DEC' --> 2025
    '06/2024' --> 2024
    '3 July 2019' --> 2019"""
    if not date_str:
        return 1900
    try:
        year_candidate = date_str.split('-')[0]
        # Check if the first part is a valid year
        if year_candidate.isdigit() and len(year_candidate) == 4:
            return int(year_candidate)
        year_candidate = date_str.split(' ')[0]
        # Check if the first part is a valid year in a different format
        if year_candidate.isdigit() and len(year_candidate) == 4:
            return int(year_candidate)
        # If the date is in a different format, try to parse it
        parsed_date = parser.parse(date_str, fuzzy=True)
        if parsed_date:
            return parsed_date.year
        else: return 1900
    except ValueError:
        return 1900

from functools import lru_cache

# Iterate over all_items and get a dict of collection name to item when type is not note neither attachment

@lru_cache(maxsize=None)
def get_collection(coll_id):
    print("Querying cache for collection id:", coll_id)
    return zot.collection(coll_id)


In [6]:
# BAD IDEA: it removes the notes that are used later...
# Keep only items where date is 2018 or later
#articles_to_review = [item for item in all_items if get_year_from_date(item['data'].get('date', '')) >= 2018]

In [7]:
# print(len(articles_to_review))

1259


# Count the number of occurence of papers in different query results (each query is a collection in this case)

## First explore collection to see duplicates in titles

In [8]:
# Find all items with a duplicate title
from collections import defaultdict

title_counts = defaultdict(int)
for item in all_items:
    # Only consider items with itemType ≠ 'note' or 'attachment'
    if item['data']['itemType'] in ['note', 'attachment']:
        continue
    title = item['data']['title']
    title_counts[title] += 1
# Print titles that have duplicates
for title, count in title_counts.items():
    if count > 1:
        print(f"Title: {title} | Count: {count}")



Title: Neural Machine Translation and Linked Data | Count: 2


## Updating articles to review with the number of collections they belong to as a tag


In [9]:
# For each item, count the number of Collections it belongs in add it as a tag into the entry

def cleanup_collection_count_tags(articles_to_review):
    """Remove existing #collections:N tags from all items."""
    for item in articles_to_review:
        # Only consider items with itemType ≠ 'note' or 'attachment'
        if item['data']['itemType'] in ['note', 'attachment']:
            continue
        tags = item['data'].get('tags', [])
        new_tags = [t for t in tags if not (t['tag'].startswith('#collections:'))]
        num_cleaned_up_items = 0
        if len(new_tags) != len(tags):
            item_clone = item.copy()
            item_clone['data'] = {}
            item_clone['data']['tags'] = new_tags
            zot.update_item(item_clone)
            num_cleaned_up_items += 1
        if num_cleaned_up_items % 100 == 0:
            print(f"Cleaned up {num_cleaned_up_items} items.")
    print(f"{num_cleaned_up_items} items cleaned up.")

def update_collection_count_tags(articles_to_review):
    """Update each item's tags with the number of collections it belongs to."""
    hold = True
    for item in articles_to_review:
        # Only consider items with itemType ≠ 'note' or 'attachment'
        if item['data']['itemType'] in ['note', 'attachment']:
            continue
         # Check if the item has a date and if it is 2018 or later
        date = item['data'].get('date', '')
        if date and get_year_from_date(date) < 2018:
            continue
        collections = item.get('data', {}).get('collections', [])
        collections = [get_collection(c)['data']['name'] for c in collections]
        # filter out collections beginning with '@'
        collections = [c for c in collections if not c.startswith('@')]
        tag_value = f"#collections:{len(collections)}"
        # Add the tag to the item if it doesn't already exist
        tag_names = [t['tag'] for t in item['data'].get('tags', [])]
        if tag_value not in tag_names:
            print("Updating item with key:", item['data']['key'], "with tag:", tag_value)
            zot.add_tags(item, tag_value)


# cleanup_collection_count_tags(all_items)
update_collection_count_tags(all_items)


Querying cache for collection id: FCSC39F3
Querying cache for collection id: 7QNUW22V
Querying cache for collection id: IB7678WJ
Querying cache for collection id: PUYL5DFJ
Querying cache for collection id: A8KESFDY
Querying cache for collection id: PRCHD7T2
Querying cache for collection id: RK4WUSHU
Querying cache for collection id: 5DTWJ5PL
Querying cache for collection id: WRBFCTN7
Querying cache for collection id: 647NIJ7Q
Querying cache for collection id: R4CWAJBV
Querying cache for collection id: 7UV7DUD2
Querying cache for collection id: IN5LFKRB
Querying cache for collection id: JCRHATRI
Querying cache for collection id: 7PGNLZ82
Querying cache for collection id: Q63YJVGE
Querying cache for collection id: 36UYP5RM
Querying cache for collection id: 7IMHTGMR
Querying cache for collection id: RBEQ8VGR
Querying cache for collection id: 2S2JJRF7
Querying cache for collection id: 3X6IXK99
Querying cache for collection id: ELK3XH49
Updating item with key: 7Y9WS35N with tag: #collection

# Assign papers to the different reviewers, keeping only items from 2018 and later


In [10]:

reviewers = [
    "@Mike",
    "@Gilles",
    "@Ana",
    "@Dagmar",
    "@Ciprian",
    "@Elena",
    "@Buket"]


def assign_items_to_reviewers():
    """Assign items from 2018 and later to reviewers in a round-robin fashion."""
    global reviewers
    # Create a dictionary to hold the items for each reviewer
    reviewer_items = {reviewer: [] for reviewer in reviewers}
    # Filter items by year and assign to reviewers
    item_number = 0
    for item in all_items:
        # Only consider items with itemType ≠ 'note' or 'attachment'
        if item['data']['itemType'] in ['note', 'attachment']:
            continue
        # Check if the item has a date and if it is 2018 or later
        date = item['data'].get('date', '')
        if date and get_year_from_date(date) >= 2018:
            item_number += 1
            # Assign the item to a reviewer in a round-robin fashion
            reviewer = reviewers[item_number % len(reviewers)]
            reviewer_items[reviewer].append(item)

    # Print the number of items assigned to each reviewer
    for reviewer, ritems in reviewer_items.items():
        print(f"{reviewer}: {len(ritems)} items")

    # Then put the items into the corresponding collection using Zotero API
    for reviewer, ritems in reviewer_items.items():
        # Create a collection for the reviewer if it doesn't exist
        collection_name = f"{reviewer}"
        collections = zot.collections()
        collection = next((c for c in collections if c['data']['name'] == collection_name), None)
        if not collection:
            collection = zot.create_collection([{'name': collection_name}])
            if collection['success']['0']:
                collection_id = collection['success']['0']
            else:
                print(f"Failed to create collection {collection_name}")
                continue
        else:
            collection_id = collection['data']['key']

        # Add items to the collection
        for item in ritems:
            zot.addto_collection(collection_id, item)
            print(f"Added item {item['data']['title']} to collection {collection_name}")


In [11]:
# assign_items_to_reviewers()

# Add a note to each item in the collection


In [6]:
note_html = """<div data-schema-version="9"><p>[T3.3 REVIEW]</p>
<ul>
<li>
Does it talk about KG AND MT (yes/no):
</li>
<li>
Direction of influence (KG_for_MT/MT_for_KG/INDIRECT):
</li>
<li>
Relevant (include it in next pass or just forget it, yes/no/maybe):
</li>
<li>
—
</li>
<li>
Kind of KG (Ontology/Language/Multilingual/…):
</li>
<li>
Kind of MT (RBMT/EBMT/SMT/NMT/LLM/OTHER):
</li>
</ul>
<p>Remark (if applicable): </p>
</div>
"""


In [7]:
def chunk_list(lst, chunk_size):
    for i in range(0, len(lst), chunk_size):
        yield lst[i:i + chunk_size]

In [8]:
# SHOULD BE RUN ONLY ONCE, OTHERWISE IT CREATES DUPLICATE NOTES. It adds a note with the content of note_html to each item in the library that is not a note nor an attachment, as a child of the item.
def add_note_templates(items=None):
    """Add a note template to each item in the library."""
    items = items or all_items
    notes = []
    for item in items:
        # Only consider items with itemType ≠ 'note' or 'attachment'
        if item['data']['itemType'] in ['note', 'attachment']:
            continue
        # Create a note for the item
        note = {
            'itemType': 'note',
            'note': note_html,
            'parentItem': item['data']['key'],
        }
        notes.append(note)

    for chunk in chunk_list(notes, 40):
        zot.create_items(chunk)
        print(f"Created {len(chunk)} notes")


# Add a note template to all elements
# add_note_templates()
# add_note_templates(items=collection_to_items.get('@Added A Posteriori (Semantic Scholar)'))

# Explore the current items and add tags to follow reviewing work advancement

In [8]:
# Update the items in dirty_item_keys and their children in all_items with their new version from Zotero to avoid updating the same item multiple times and raise exceptions because of it
def update_dirty_items(dirty_item_keys: list[str]):
    global all_items
    updated_items = [zot.item(key) for key in dirty_item_keys]
    updated_items_children = []
    for item in updated_items:
        children = zot.children(item['data']['key'])
        updated_items_children.extend(children)
    updated_items.extend(updated_items_children)
    dirty_item_keys = [item['data']['key'] for item in updated_items]
    all_items = [e for e in all_items if e["key"] not in dirty_item_keys] + updated_items


In [9]:
import re
from collections import defaultdict
from typing import Mapping, Any


def get_item_by_key(key):
    global all_items
    """Return the item dict from all_items whose data.key == key, or None if not found."""
    return next((it for it in all_items if it.get('key') == key), None)

def get_notes_of_item(item):
    """Return a list of attachment items that are children of the given item."""
    global all_items
    if item is None:
        return None
    item_key = item if isinstance(item, str) else item.get('key') if isinstance(item, Mapping) else None
    if item_key is None:
        return None
    attachments = [it for it in all_items if it.get('data', {}).get('itemType') == 'note' and it.get('data', {}).get('parentItem') == item_key]
    return attachments

def extract_answer_from_note(note_text, param, unanswered_hints=None):
    """
    Return the answer (lowercased) on the line that contains `param` (or any of the params).
    If `param` is a list/tuple, try each value in order and return the first match.
    Priority:
      1. Text after a colon or semicolon on that line.
      2. Text inside the last parentheses on that line.
      3. Any trailing text after the parameter token.
    Returns None if no answer is found.
    """
    if not note_text:
        return None

    if unanswered_hints is None:
        unanswered_hints = ['yes/no']

    # Normalize param into a list of non-empty stripped strings
    if isinstance(param, (list, tuple)):
        params = [(p or '').strip() for p in param]
    else:
        params = [ (param or '').strip() ]
    params = [p for p in params if p]

    if not params:
        return None

    lines = note_text.splitlines()
    for i, line in enumerate(lines):
        low_line = line.lower()
        for p in params:
            low_p = p.lower()
            if low_p in low_line:
                # 1) after colon or semicolon
                m = re.search(r'[:;]\s*(.+)$', low_line)
                if m:
                    ans = re.sub(r'&nbsp;', ' ', m.group(1)).strip()
                    if ans:
                        if ans == '</p>':
                            # try the next line
                            if i + 1 < len(lines):
                                next_line = lines[i + 1].strip().lower()
                                if next_line:
                                    ans = next_line
                        return re.sub(r'</?p>', '', ans).strip()
                    else:
                        return 'unanswered'

                # 2) text inside the last parentheses on the line
                matches = re.findall(r'\(([^)]*?)\)', low_line)
                if matches:
                    ans = re.sub(r'&nbsp;', ' ', matches[-1]).strip()
                    if ans:
                        if ans in unanswered_hints:
                            return 'unanswered'
                        return ans

                # 3) trailing text after the parameter token
                idx = low_line.find(low_p)
                if idx != -1:
                    rest = re.sub(r'&nbsp;', ' ', low_line[idx + len(p):]).strip()
                    if rest:
                        return rest
                return 'unanswered'

    return None


In [10]:


def decode_review_note(note_text):
    # Scan the note text to get answers to the 3 first lines
    kg_and_mt = extract_answer_from_note(note_text, "Does it talk about KG AND MT")
    influence = extract_answer_from_note(note_text, "Direction of influence", unanswered_hints=['KG_for_MT/MT_for_KG/INDIRECT'])
    relevant = extract_answer_from_note(note_text, ("Relevant (include it in next pass or just forget it, yes/no/maybe):", "Relevant (include it in next pass or just forget it", "Relevant (include it in next pass", "Relevant"), unanswered_hints=['yes/no/maybe', 'include it in next pass or just forget it, yes/no/maybe'])
    return kg_and_mt, influence, relevant

kgnmt_answers = defaultdict(set)
influence_answers = defaultdict(set)
relevance_answers = defaultdict(set)

def tag_review_status():
    """Tag items associated to a review note that is equals to the notes_html."""
    count = 0
    dirty_item_keys = []
    global kgnmt_answers, influence_answers, relevance_answers
    kgnmt_answers = defaultdict(set)
    influence_answers = defaultdict(set)
    relevance_answers = defaultdict(set)
    for item in all_items:
        # Only consider notes
        if item['data']['itemType'] in ['note']:
            # filter notes that have nothing to do with GOBLIN review
            if '[T3.3 REVIEW]' not in item['data']['note']:
                continue
            note_text = item['data']['note']
            parent_item_key = item['data'].get('parentItem')
            parent_item = get_item_by_key(parent_item_key)
            if not parent_item:
                print(f"Note {item['key']} has no parent item")
                continue
            # If parent item date is before 2018, skip it
            parent_item_date = parent_item['data'].get('date', '')
            if parent_item_date and get_year_from_date(parent_item_date) < 2018:
                continue
            parent_item_tags = [t["tag"] for t in parent_item['data'].get('tags', [])]
            parent_item_is_dirty = False
            if note_text.strip() != note_html.strip():
                # Note has been modified, tag the item as '#reviewed'
                count += 1
                # print(f"Item {parent_item_key} has note {note_text}")
                kgnmt, influence, relevance = decode_review_note(note_text)
                kgnmt_answers[kgnmt].add(parent_item_key)
                influence_answers[influence].add(parent_item_key)
                relevance_answers[relevance].add(parent_item_key)
                if '#reviewed' not in parent_item_tags:
                    parent_item_tags.append('#reviewed')
                    parent_item_is_dirty = True
                if '#to_review' in parent_item_tags:
                    parent_item_tags.remove('#to_review')
                    parent_item_is_dirty = True
            else:
                # Note is unchanged, tag the item as '#to_review'
                if '#to_review' not in parent_item_tags:
                    parent_item_tags.append('#to_review')
                    parent_item_is_dirty = True
            if parent_item_is_dirty:
                parent_item_clone = parent_item.copy()
                parent_item_clone["data"] = {}
                parent_item_clone["data"]["tags"] = []
                for tag in parent_item_tags:
                    parent_item_clone["data"]["tags"].append({"tag": f"{tag}"})
                print("Updating item", parent_item_key, "with tags:", parent_item_tags)
                zot.update_item(parent_item_clone)
                dirty_item_keys.append(parent_item_clone["key"])
    print(f"{count} items with review notes found.")
    print(f"{len(dirty_item_keys)} items to update.")
    update_dirty_items(dirty_item_keys)


In [19]:

tag_review_status()

print("kgnmt answers:", kgnmt_answers)
print("influence answers:", influence_answers)
print("relevance answers:", relevance_answers)

Updating item QMPQIZ8T with tags: ['#reviewed']
1191 items with review notes found.
1 items to update.
kgnmt answers: defaultdict(<class 'set'>, {'unanswered': {'BGLA2T4P', 'Q6HH4AL3', 'JIS4KCBF', '79I32RFQ', 'TJIXGFHA', 'GLPS6YXR', 'KLH85E8Z', 'G7Z39WCU', 'KY89ZZHT', 'RP76EZ8B', '8KPVIE2K', 'QMPQIZ8T', 'G8WJN36M', 'A5B8P3JX', 'MWYHD6ZP', 'M4BH9AL7', '4I3VISCG', 'T7JZK6VW', 'FHKJN685', 'C4ZBN758', 'MECLG9QW', '7GE4ZX9X', 'E2YPXXJ4', 'THUB9JY3', 'D4IEWSMU', '4FISYZ4G', 'UJJQYPBM', 'CV4R4ESX', 'L98WC7EG', 'NG7HZ8IJ'}, 'no': {'LTD3GIGD', '2SMZGJWE', '6MQLN9ND', 'IQ3H4A5X', 'YBHAY7JR', 'CKHF98DY', 'JSYL7MZK', 'V2FC5D4X', 'RL6GR3GK', 'YXZIGWVA', '9BJ3BQAZ', '8JS99HV3', 'SLFEMBJI', '7XVSHAST', 'GSUMU8G6', '86GHE4MI', '3TSKKS5V', 'IRXWKLID', '2DF6HIUH', '6ZM5HPYD', 'TPU4TFND', 'XF854FGG', 'EQQTF5S4', 'FBA7WX3J', '27WJ8WX3', 'R9MCS24Y', 'UWUUXIX3', '7N85EH9E', 'IQKAMABX', '95NYUAY5', '59VRC8EL', 'SNQA3XQ7', 'XS5Q5MET', 'MMLSBLPR', '24BRSDUI', 'JTJV2LKN', '6L4N8T4Y', 'VB83EFG6', 'XBYI79GN', '9L

## Tagging the reviewed items according to the note

Warning: this will raise an exception if you modify an item that has been modified before. Hence you have to reload all_items and reprocess them for review status until the set is stable (no change in tags) before tagging for another tag.

In [20]:

def tag_items_with_tag(item_keys, tag):
    dirty_item_keys = []
    for key in item_keys:
        item = get_item_by_key(key)
        if not item:
            continue
        item_tags = [t["tag"] for t in item['data'].get('tags', [])]
        if tag not in item_tags:
            item_tags.append(tag)
            item_clone = item.copy()
            item_clone["data"] = {}
            item_clone["data"]["tags"] = []
            for t in item_tags:
                item_clone["data"]["tags"].append({"tag": f"{t}"})
            print("Updating item", key, "with tags:", item_tags)
            zot.update_item(item_clone)
            dirty_item_keys.append(item_clone["key"])
    update_dirty_items(dirty_item_keys)




In [21]:

tag_items_with_tag(kgnmt_answers['yes'], '#kg_and_mt_yes')
# tag_items_with_tag(influence_answers['kg_for_mt'], '#kg_for_mt')
# tag_items_with_tag(influence_answers['mt_for_kg'], '#mt_for_kg
# tag_items_with_tag(relevance_answers['yes'], '#t33_relevant')
# tag_items_with_tag(relevance_answers['no'], '#not_relevant')
# tag_items_with_tag(relevance_answers['maybe'], '#t33_relevant_maybe')


In [22]:
tag_items_with_tag(relevance_answers['yes'], '#t33_relevant')


In [23]:
tag_items_with_tag(relevance_answers['maybe'], '#t33_relevant_maybe')

In [24]:
# Additionally, tag unclear answer by interpreting them
interpreted_answers = defaultdict(set)

for relevance, ids in relevance_answers.items():
    if relevance in ['yes', 'no', 'maybe', 'unanswered']: continue
    if 'maybe' in relevance:
        tag = '#t33_relevant_maybe'
    elif 'yes' in relevance:
        if 'but' in relevance or 'however' in relevance:
            tag = '#t33_relevant_maybe'
        else:
            tag = '#t33_relevant'
    else:
        continue
    interpreted_answers[tag].update(ids)

for tag, ids in interpreted_answers.items():
    tag_items_with_tag(ids, tag)


# Debug stuff, just ignore it

In [48]:
entry = get_notes_of_item('7L2QVQA3')
extract_answer_from_note(entry[0]['data']['note'], ("Relevant (include it in next pass or just forget it, yes/no/maybe):"))



'include it in next pass or just forget it, yes/no/maybe'

In [16]:
# cleanup notes that have no parent item and are equals to the default note_html
def cleanup_orphaned_notes():
    count = 0
    for item in all_items:
        # Only consider notes
        if item['data']['itemType'] in ['note']:
            # filter notes that have nothing to do with GOBLIN review
            if '[T3.3 REVIEW]' not in item['data']['note']:
                continue
            note_text = item['data']['note']
            parent_item_key = item['data'].get('parentItem')
            parent_item = get_item_by_key(parent_item_key)
            if not parent_item and note_text.strip() == note_html.strip():
                # Delete the note
                zot.delete_item(item['data'])
                count += 1
    print(f"Deleted {count} orphaned notes.")

cleanup_orphaned_notes()

Deleted 1747 orphaned notes.


In [15]:
zot.collections()

[{'key': 'WJ345NH6',
  'version': 26772,
  'library': {'type': 'group',
   'id': 6004137,
   'name': 'KG_and_MT_GOBLIN_T3.3',
   'links': {'alternate': {'href': 'https://www.zotero.org/groups/6004137',
     'type': 'text/html'}}},
  'links': {'self': {'href': 'https://api.zotero.org/groups/6004137/collections/WJ345NH6',
    'type': 'application/json'},
   'alternate': {'href': 'https://www.zotero.org/groups/6004137/collections/WJ345NH6',
    'type': 'text/html'}},
  'meta': {'numCollections': 0, 'numItems': 0},
  'data': {'key': 'WJ345NH6',
   'version': 26772,
   'name': '@Oğuzhan',
   'parentCollection': False,
   'relations': {}}},
 {'key': '5F2FXN8F',
  'version': 26003,
  'library': {'type': 'group',
   'id': 6004137,
   'name': 'KG_and_MT_GOBLIN_T3.3',
   'links': {'alternate': {'href': 'https://www.zotero.org/groups/6004137',
     'type': 'text/html'}}},
  'links': {'self': {'href': 'https://api.zotero.org/groups/6004137/collections/5F2FXN8F',
    'type': 'application/json'},
  

In [12]:
# print all item types in the library
item_types = set()
for item in all_items:
    item_types.add(item['data']['itemType'])
print(item_types)

{'preprint', 'thesis', 'conferencePaper', 'webpage', 'book', 'attachment', 'document', 'journalArticle', 'note', 'report', 'bookSection'}


In [22]:


collection_to_items = defaultdict(list)
collections_of_reviewers = set()
for item in all_items:
    if item['data']['itemType'] in ['note', 'attachment']:
        continue
    # If parent item date is before 2018, skip it
    item_date = item['data'].get('date', '')
    if item_date and get_year_from_date(item_date) < 2018:
        continue
    collection_ids = item.get('data', {}).get('collections', [])
    has_a_collection = False
    for collection_id in collection_ids:
        collection = get_collection(collection_id)
        collection_name = collection['data']['name']
        if not collection_name.startswith('@'):
            continue
        collections_of_reviewers.add(collection_name)
        has_a_collection = True
        #print(f"Item {item['data']['key']} belongs to collection {collection_name}")
        collection_to_items[collection_name].append(item)
    if not has_a_collection:
        collection_to_items['<no-collection>'].append(item)
print({k: len(v) for k, v in collection_to_items.items()})



Querying cache for collection id: 5F2FXN8F
Querying cache for collection id: RK4WUSHU
Querying cache for collection id: 36UYP5RM
Querying cache for collection id: PGRN7559
Querying cache for collection id: IB7678WJ
Querying cache for collection id: 7UV7DUD2
Querying cache for collection id: DRGETMYG
Querying cache for collection id: R4CWAJBV
Querying cache for collection id: X28T6897
Querying cache for collection id: MPIJXLWR
Querying cache for collection id: FCSC39F3
Querying cache for collection id: PRCHD7T2
Querying cache for collection id: A8KESFDY
Querying cache for collection id: WRBFCTN7
Querying cache for collection id: 7IMHTGMR
Querying cache for collection id: RBEQ8VGR
Querying cache for collection id: D8NBXCUT
Querying cache for collection id: JCRHATRI
Querying cache for collection id: PUYL5DFJ
Querying cache for collection id: 5DTWJ5PL
Querying cache for collection id: Q63YJVGE
Querying cache for collection id: BPS8B4G6
Querying cache for collection id: IN5LFKRB
Querying ca

In [23]:
print([c.get('key') for c in collection_to_items.get('<no-collection>')])


['7Y9WS35N']


In [75]:
# For each pair of collection_of_reviewer, Check common items in each two collections pairs
from itertools import combinations
for coll1, coll2 in combinations(collections_of_reviewers, 2):
    items1 = collection_to_items[coll1]
    items2 = collection_to_items[coll2]
    keys1 = set([item['data']['key'] for item in items1])
    keys2 = set([item['data']['key'] for item in items2])
    common_keys = keys1.intersection(keys2)
    if common_keys:
        print(f"Collections {coll1} and {coll2} have {len(common_keys)} common items: {common_keys}")


Collections @Mehwish and @Miguel have 1 common items: {'Y7JZZ7CB'}


In [6]:
# retrieve the first note in items and display it
notes = [item for item in all_items if item['data']['itemType'] == 'note']

In [20]:
print(zot.item_types())

[{'itemType': 'artwork', 'localized': 'Artwork'}, {'itemType': 'audioRecording', 'localized': 'Audio Recording'}, {'itemType': 'bill', 'localized': 'Bill'}, {'itemType': 'blogPost', 'localized': 'Blog Post'}, {'itemType': 'book', 'localized': 'Book'}, {'itemType': 'bookSection', 'localized': 'Book Section'}, {'itemType': 'case', 'localized': 'Case'}, {'itemType': 'conferencePaper', 'localized': 'Conference Paper'}, {'itemType': 'dataset', 'localized': 'Dataset'}, {'itemType': 'dictionaryEntry', 'localized': 'Dictionary Entry'}, {'itemType': 'document', 'localized': 'Document'}, {'itemType': 'email', 'localized': 'E-mail'}, {'itemType': 'encyclopediaArticle', 'localized': 'Encyclopedia Article'}, {'itemType': 'film', 'localized': 'Film'}, {'itemType': 'forumPost', 'localized': 'Forum Post'}, {'itemType': 'hearing', 'localized': 'Hearing'}, {'itemType': 'instantMessage', 'localized': 'Instant Message'}, {'itemType': 'interview', 'localized': 'Interview'}, {'itemType': 'journalArticle', '